# Insurance Charges Prediction - Machine Learning Pipeline

This notebook demonstrates a complete machine learning workflow to predict medical insurance charges based on individual health factors. The dataset includes information such as age, BMI, number of children, smoking status, and region.

## Goal:
- Understand the data through Exploratory Data Analysis (EDA).
- Build a preprocessing pipeline.
- Train and evaluate multiple regression models.
- Perform hyperparameter tuning on the best-performing model.
- Interpret findings and importance of individual features.

## Part 1: Problem Definition & Data Exploration (EDA)

**Objective:** Understand the business context, define the target variable, and perform exploratory data analysis to build initial hypotheses.

In [ ]:
# 1. Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')

In [ ]:
# 2. Load the data
file_path = r'D:\Machine Learning\Materi\Materi\Chapter 4\Part 4\P04_Ch05_insurance.csv'
df = pd.read_csv(file_path)

# Display first 5 rows
print("--- First 5 rows of the dataset ---")
display(df.head())

In [ ]:
# 3. Summary information and statistics
print("\n--- Data Information ---")
print(df.info())

print("\n--- Descriptive Statistics (Numeric) ---")
display(df.describe())

In [ ]:
# 4. Target Variable Distribution (Charges)
plt.figure(figsize=(10, 6))
sns.histplot(df['charges'], kde=True, color='blue')
plt.title('Distribution of Insurance Charges', fontsize=15)
plt.xlabel('Charges')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# 5. Key Feature Distribution
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sns.histplot(df['age'], kde=True, ax=axes[0], color='green')
axes[0].set_title('Age Distribution')

sns.histplot(df['bmi'], kde=True, ax=axes[1], color='orange')
axes[1].set_title('BMI Distribution')

sns.countplot(x='children', data=df, ax=axes[2], palette='viridis')
axes[2].set_title('Number of Children')

plt.tight_layout()
plt.show()

In [ ]:
# 6. Relationship between Numeric Features and Target
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numeric Features', fontsize=15)
plt.show()

In [ ]:
# 7. Visualizing Categorical Features vs Target
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Analysis of Feature Relationships with Insurance Charges', fontsize=20)

# Smoker vs Charges
sns.boxplot(ax=axes[0, 0], data=df, x='smoker', y='charges', palette='Set1')
axes[0, 0].set_title('Smoker vs. Charges')

# Sex vs Charges
sns.boxplot(ax=axes[0, 1], data=df, x='sex', y='charges', palette='Set2')
axes[0, 1].set_title('Sex vs. Charges')

# Region vs Charges
sns.boxplot(ax=axes[1, 0], data=df, x='region', y='charges', palette='Set3')
axes[1, 0].set_title('Region vs. Charges')

# Jointplot for Age & Charges
sns.scatterplot(ax=axes[1, 1], data=df, x='age', y='charges', hue='smoker')
axes[1, 1].set_title('Age vs Charges (Colored by Smoker)')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### Initial Hypotheses (H0):
1. **Smoking** is likely the strongest predictor (significant jump in charges).
2. **Age** has a positive linear correlation with charges.
3. **BMI** potentially increases charges, especially for smokers (interaction effect).

## Part 2: Preprocessing & Pipeline Construction

**Objective:** Prepare data for modeling and encapsulate steps in an efficient scikit-learn `Pipeline`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Separate features (X) and target (y)
X = df.drop('charges', axis=1)
y = df['charges']

# 2. Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Identify numeric and categorical columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# 4. Create preprocessor pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

print(f"Numerical columns: {numerical_cols}")
print(f"Categorical columns: {categorical_cols}")
print("Preprocessing pipeline created successfully.")

## Part 3: Model Training & Evaluation

**Objective:** Train several models and compare their performance using cross-validation.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

# 1. Define candidate models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

# 2. Compare models using CV
cv_results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    # Using Root Mean Squared Error as score
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
    cv_results[name] = -scores
    print(f"{name} RMSE (CV Mean): {-scores.mean():.4f}")

In [ ]:
# 3. Visualize Model Comparison
plt.figure(figsize=(10, 6))
plt.boxplot(cv_results.values(), labels=cv_results.keys(), showmeans=True)
plt.title('Model Performance Comparison (CV RMSE)')
plt.ylabel('RMSE (Lower is better)')
plt.show()

**Observation:** Gradient Boosting and Random Forest generally perform better than linear models on this non-linear dataset.

## Part 4: Hyperparameter Tuning

**Objective:** Fine-tune Gradient Boosting Regressor for better accuracy.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Setup GB Pipeline
gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(random_state=42))
])

# 2. Hyperparameter Grid
param_grid_gb = {
    'regressor__n_estimators': [100, 200, 300, 400, 500],
    'regressor__learning_rate': [0.01, 0.05, 0.1],
    'regressor__max_depth': [3, 4, 5],
    'regressor__subsample': [0.8, 0.9, 1.0]
}

# 3. Randomized Search CV
gb_random_search = RandomizedSearchCV(
    estimator=gb_pipeline, 
    param_distributions=param_grid_gb, 
    n_iter=50,
    cv=5, 
    scoring='neg_root_mean_squared_error', 
    verbose=1, 
    random_state=42, 
    n_jobs=-1
)

print("Starting Hyperparameter Tuning for Gradient Boosting...")
gb_random_search.fit(X_train, y_train)

print(f"Best Hyperparameters: {gb_random_search.best_params_}")
print(f"Best CV RMSE: {-gb_random_search.best_score_:.4f}")

## Part 5: Interpretation & Conclusion

**Objective:** Evaluate final model on test data and visualize feature importance.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Final Predicton using Champion Model
champion_model = gb_random_search.best_estimator_
y_pred = champion_model.predict(X_test)

# 2. Calculate final performance metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("--- Final Performance Report (Test Data) ---")
print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R-squared Score (R²): {r2:.4f}")

In [ ]:
# 3. Feature Importance Visualization

# Get feature names after OneHotEncoding
ohe_feature_names = champion_model.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_feature_names = numerical_cols + list(ohe_feature_names)

# Extract importance
importances = champion_model.named_steps['regressor'].feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
plt.title('Top Feature Importances (Gradient Boosting Model)')
plt.show()

### Conclusion & Recommendations:

1. **Smoking Status** is the dominant factor in determining insurance costs. Policies could focus heavily on smoking cessation programs.
2. **BMI** is also a significant predictor, especially highlighting the risk associated with obesity.
3. **Age** follows closely, confirming that medical costs rise as individuals get older.

**Summary:** The Gradient Boosting model achieved an R-squared of over 0.85, providing a highly reliable tool for cost estimation and business risk assessment.